In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from io import BytesIO
from minio import Minio

In [ ]:
MINIO_ENDPOINT = "localhost:9000"
MINIO_BUCKET = "cgmacros"
PATIENTS = ["CGMacros-012", "CGMacros-039"]

In [ ]:
client = Minio(MINIO_ENDPOINT, access_key="minioadmin", secret_key="minioadmin", secure=False)

In [ ]:
def load_parquet(patient_id):
    path = f"silver/CGMacros/{patient_id}/{patient_id}.parquet"
    response = client.get_object(MINIO_BUCKET, path)
    df = pd.read_parquet(BytesIO(response.read()))
    response.close()
    response.release_conn()
    return df

In [ ]:
for pid in PATIENTS:
    df = load_parquet(pid)

    print(f"\n{'='*20} EDA: {pid} {'='*20}")

    # --- A. Panorama Geral ---
    print("\n1. Estrutura e Nulos:")
    print(df.info())
    print(f"\nValores ausentes:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

    # --- B. Estatística Descritiva ---
    print("\n2. Resumo Estatístico (Glicose e Batimentos):")
    cols_bio = ['Libre GL', 'Dexcom GL', 'HR']
    print(df[cols_bio].describe())

    # --- C. Visualizações ---
    plt.figure(figsize=(15, 10))

    # Gráfico 1: Série Temporal de Glicose
    plt.subplot(2, 1, 1)
    sns.lineplot(data=df, x='Timestamp', y='Libre GL', label='Libre GL', color='blue', alpha=0.7)
    sns.lineplot(data=df, x='Timestamp', y='Dexcom GL', label='Dexcom GL', color='red', alpha=0.5)
    plt.title(f'Tendência de Glicose ao Longo do Tempo - {pid}')
    plt.ylabel('mg/dL')
    plt.legend()

    # Gráfico 2: Distribuição (Histograma)
    plt.subplot(2, 2, 3)
    sns.histplot(df['Libre GL'].dropna(), kde=True, color='blue')
    plt.title('Distribuição Glicose (Libre)')

    # Gráfico 3: Correlação entre Sensores
    plt.subplot(2, 2, 4)
    if df['Libre GL'].notnull().any() and df['Dexcom GL'].notnull().any():
        sns.scatterplot(data=df, x='Libre GL', y='Dexcom GL', alpha=0.3)
        plt.title('Libre vs Dexcom (Consistência)')

    plt.tight_layout()
    plt.show()

    # --- D. Análise de Refeições (se houver dados) ---
    if 'Meal Type' in df.columns:
        print("\n3. Impacto das Refeições na Glicose Média:")
        meal_impact = df.groupby('Meal Type')['Libre GL'].mean().sort_values(ascending=False)
        print(meal_impact)

In [ ]:
# 1. Preencher pequenos buracos nos sensores (até 15 min) por interpolação
df['Libre GL'] = df['Libre GL'].interpolate(method='linear', limit=3)
df['HR'] = df['HR'].interpolate(method='linear', limit=5)

# 2. Criar uma coluna 'Is_Postprandial' (Janela de 2h após comer)
# Isso ajuda a isolar o efeito da comida na glicose
df['Meal_Event'] = df['Meal Type'].notnull().astype(int)
df['In_Meal_Window'] = df['Meal_Event'].rolling(window=24, min_periods=1).max() # Assumindo leitura a cada 5min

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO, StringIO

In [ ]:
HORIZONTE = 30  # Agora 30 minutos reais (visto que o intervalo é 1 min)

def prepare_for_prediction(df, horizonte_minutos=30):
    df = df.sort_values('Timestamp').copy()

    # --- 1. Janela de Alimentação ---
    # Se o dado é 1 min, 2 horas = 120 períodos (minutos)
    df['Meal_Event'] = df['Meal Type'].notnull().astype(int)
    df['In_Meal_Window'] = df['Meal_Event'].rolling(window=120, min_periods=1).max().fillna(0)

    # --- 2. Target (Previsão para daqui a 30 min) ---
    df['target_glucose'] = df['Libre GL'].shift(-horizonte_minutos)

    # --- 3. Lags (Passado em minutos reais) ---
    # Criando atrasos para 5, 15, 30 e 60 minutos atrás
    for lag in [5, 15, 30, 60]:
        df[f'lag_GL_{lag}min'] = df['Libre GL'].shift(lag)
        df[f'lag_HR_{lag}min'] = df['HR'].shift(lag)

    # --- 4. Tendência (Diferença de curto prazo) ---
    # Variação nos últimos 5 e 10 minutos
    df['diff_5min'] = df['Libre GL'].diff(5)
    df['diff_10min'] = df['Libre GL'].diff(10)

    # --- 5. Variáveis Cíclicas ---
    df['hour'] = df['Timestamp'].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/23.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/23.0)

    # Limpeza: Remover linhas sem target ou sem os lags básicos
    # Com horizonte de 30 e lag de 60, perderemos as primeiras 60 e as últimas 30 linhas.
    df = df.dropna(subset=['target_glucose', 'lag_GL_5min', 'lag_GL_60min'])

    return df

In [ ]:
def save_to_minio(data_buffer, path, content_type):
    data_buffer.seek(0)
    client.put_object(
        MINIO_BUCKET, path,
        data=data_buffer,
        length=data_buffer.getbuffer().nbytes if hasattr(data_buffer, 'getbuffer') else len(data_buffer.getvalue()),
        content_type=content_type
    )

In [ ]:
for pid in PATIENTS:
    print(f"🚀 Processando e salvando GOLD para: {pid}")

    try:
        # A. Carregar Silver
        source_path = f"silver/CGMacros/{pid}/{pid}.parquet"
        response = client.get_object(MINIO_BUCKET, source_path)
        df = pd.read_parquet(BytesIO(response.read()))
        response.close()

        # B. Feature Engineering com Horizonte de 30 min
        df_featured = prepare_for_prediction(df, horizonte_minutos=HORIZONTE)

        # C. Salvar GOLD (ML Ready)
        gold_path = f"gold/CGMacros/{pid}/{pid}_ML_READY.parquet"
        parquet_buffer = BytesIO()
        df_featured.to_parquet(parquet_buffer, index=False)
        save_to_minio(parquet_buffer, gold_path, 'application/x-parquet')

        # D. Salvar Estatísticas
        stats_path = f"gold/metadata/{pid}/{pid}_stats_summary.csv"
        csv_buffer = StringIO()
        df_featured.describe().rename_axis('statistic').to_csv(csv_buffer)
        save_to_minio(BytesIO(csv_buffer.getvalue().encode()), stats_path, 'text/csv')

        # E. Salvar Gráfico de Validação (Target vs Atual)
        plt.figure(figsize=(12, 6))
        # Plotando apenas um recorte para visibilidade
        subset = df_featured.iloc[500:1000]
        plt.plot(subset['Timestamp'], subset['Libre GL'], label='Glicose Atual (t)', alpha=0.8)
        plt.plot(subset['Timestamp'], subset['target_glucose'], label='Alvo (t + 30min)', linestyle='--', color='red')
        plt.fill_between(subset['Timestamp'], 100, 200, where=subset['In_Meal_Window']==1,
                         color='orange', alpha=0.1, label='Janela Pós-Prandial')
        plt.title(f"Validação de Features Gold - {pid}")
        plt.legend()

        img_buffer = BytesIO()
        plt.savefig(img_buffer, format='png')
        plt.close()
        save_to_minio(img_buffer, f"gold/metadata/{pid}/{pid}_validation_gold.png", 'image/png')

        print(f"✅ {pid} concluído.\n")

    except Exception as e:
        print(f"❌ Erro ao processar {pid}: {e}")